In [1]:
import pandas as pd
import numpy as np

history = pd.read_csv(
    "../data/processed/asset_history_simulated.csv",
    parse_dates=["snapshot_date"]
)

failures = pd.read_csv(
    "../data/processed/failures_simulated.csv",
    parse_dates=["failure_date"]
)

print("History shape:", history.shape)
print("Failures shape:", failures.shape)

History shape: (92000, 10)
Failures shape: (1048, 7)


In [2]:
history = history.sort_values(
    ["asset_id", "snapshot_date"]
).reset_index(drop=True)

failures = failures.sort_values(
    ["asset_id", "failure_date"]
).reset_index(drop=True)

history.head()

,asset_id,section_id,department,asset_type,snapshot_date,asset_age_years,condition_score,criticality,usage_factor,weather_stress
0,ASSET00001,RTM-VAD-01,TRACTION,OHE,2019-01-01,8,93.126212,6,0.565548,0.416271
1,ASSET00001,RTM-VAD-01,TRACTION,OHE,2019-02-01,8,92.456850,6,0.565548,0.087173
2,ASSET00001,RTM-VAD-01,TRACTION,OHE,2019-03-01,8,95.602455,6,0.565548,0.899679
3,ASSET00001,RTM-VAD-01,TRACTION,OHE,2019-04-01,8,96.610857,6,0.565548,0.408667
4,ASSET00001,RTM-VAD-01,TRACTION,OHE,2019-05-01,8,94.519420,6,0.565548,0.391072


In [17]:
history["historical_failure_count"] = 0
history["historical_downtime_hours"] = 0.0
history["days_since_last_failure"] = np.nan

In [18]:
for asset_id, group in history.groupby("asset_id"):
    
    asset_failures = failures[
        failures["asset_id"] == asset_id
    ].sort_values("failure_date")
    
    for idx in group.index:
        
        snapshot_date = history.loc[idx, "snapshot_date"]
        
        past_failures = asset_failures[
            asset_failures["failure_date"] < snapshot_date
        ]
        
        history.loc[idx, "historical_failure_count"] = len(past_failures)
        
        history.loc[idx, "historical_downtime_hours"] = (
            past_failures["downtime_hours"].sum()
        )
        
        if len(past_failures) > 0:
            last_failure_date = past_failures["failure_date"].max()
            
            history.loc[idx, "days_since_last_failure"] = (
                snapshot_date - last_failure_date
            ).days

In [19]:
history["failure_within_30_days"] = 0

for idx in history.index:
    
    asset_id = history.loc[idx, "asset_id"]
    snapshot_date = history.loc[idx, "snapshot_date"]
    
    future_failures = failures[
        (failures["asset_id"] == asset_id) &
        (failures["failure_date"] > snapshot_date) &
        (
            failures["failure_date"]
            <= snapshot_date + pd.Timedelta(days=30)
        )
    ]
    
    if len(future_failures) > 0:
        history.loc[idx, "failure_within_30_days"] = 1

In [20]:
history["failure_within_30_days"].value_counts()

failure_within_30_days
0    90576
1      424
Name: count, dtype: int64

In [21]:
history.head()

,asset_id,section_id,department,asset_type,snapshot_date,asset_age_years,condition_score,criticality,usage_factor,weather_stress,historical_failure_count,historical_downtime_hours,days_since_last_failure,failure_within_30_days
0,ASSET00001,RTM-VAD-01,TRACTION,OHE,2019-01-01,8,93.126212,6,0.565548,0.416271,0,0.0,NaN,0
1,ASSET00001,RTM-VAD-01,TRACTION,OHE,2019-02-01,8,92.456850,6,0.565548,0.087173,0,0.0,NaN,0
2,ASSET00001,RTM-VAD-01,TRACTION,OHE,2019-03-01,8,95.602455,6,0.565548,0.899679,0,0.0,NaN,0
3,ASSET00001,RTM-VAD-01,TRACTION,OHE,2019-04-01,8,96.610857,6,0.565548,0.408667,0,0.0,NaN,0
4,ASSET00001,RTM-VAD-01,TRACTION,OHE,2019-05-01,8,94.519420,6,0.565548,0.391072,0,0.0,NaN,0


In [22]:
print("Last snapshot:", history["snapshot_date"].max())
print("Last failure:", failures["failure_date"].max())

Last snapshot: 2026-07-01 00:00:00
Last failure: 2026-08-01 00:00:00


In [23]:
latest_failure_date = failures["failure_date"].max()

history = history[
    history["snapshot_date"] + pd.Timedelta(days=30)
    <= latest_failure_date
].copy()

In [24]:
print("New last snapshot:", history["snapshot_date"].max())

New last snapshot: 2026-07-01 00:00:00


In [25]:
train = history[
    history["snapshot_date"] < "2025-01-01"
].copy()

validation = history[
    (history["snapshot_date"] >= "2025-01-01") &
    (history["snapshot_date"] < "2026-01-01")
].copy()

test = history[
    history["snapshot_date"] >= "2026-01-01"
].copy()

In [26]:
print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (72000, 14)
Validation: (12000, 14)
Test: (7000, 14)


In [27]:
print("TRAIN")
print(train["failure_within_30_days"].value_counts())
print(train["failure_within_30_days"].value_counts(normalize=True))

print("\nVALIDATION")
print(validation["failure_within_30_days"].value_counts())
print(validation["failure_within_30_days"].value_counts(normalize=True))

print("\nTEST")
print(test["failure_within_30_days"].value_counts())
print(test["failure_within_30_days"].value_counts(normalize=True))

TRAIN
failure_within_30_days
0    71684
1      316
Name: count, dtype: int64
failure_within_30_days
0    0.995611
1    0.004389
Name: proportion, dtype: float64

VALIDATION
failure_within_30_days
0    11944
1       56
Name: count, dtype: int64
failure_within_30_days
0    0.995333
1    0.004667
Name: proportion, dtype: float64

TEST
failure_within_30_days
0    6948
1      52
Name: count, dtype: int64
failure_within_30_days
0    0.992571
1    0.007429
Name: proportion, dtype: float64


In [28]:
print("Train:")
print(train["snapshot_date"].min())
print(train["snapshot_date"].max())

print("\nValidation:")
print(validation["snapshot_date"].min())
print(validation["snapshot_date"].max())

print("\nTest:")
print(test["snapshot_date"].min())
print(test["snapshot_date"].max())

Train:
2019-01-01 00:00:00
2024-12-01 00:00:00

Validation:
2025-01-01 00:00:00
2025-12-01 00:00:00

Test:
2026-01-01 00:00:00
2026-07-01 00:00:00


In [29]:
history.to_csv(
    "../data/processed/failure_ml_dataset.csv",
    index=False
)

In [30]:
train.to_csv(
    "../data/processed/failure_train.csv",
    index=False
)

validation.to_csv(
    "../data/processed/failure_validation.csv",
    index=False
)

test.to_csv(
    "../data/processed/failure_test.csv",
    index=False
)